<a href="https://colab.research.google.com/github/Pratham8765/Intellipaat/blob/main/listen_to_the_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 1: Listen to the Machine
**Find broken machines by their sound, when you only ever heard healthy ones.**

Companion to `listen_to_the_machine_instructor_script.md`. **Section numbers here = Part numbers in the script.**

- Data: DCASE 2026 Challenge Task 2 development set ([Zenodo 19336329](https://zenodo.org/records/19336329)). We use **one machine: `valveEmu`**, plus `bearingEmu` as "some other machine" in Section 8.
- Runs on **Kaggle, GPU T4, Internet ON** (Settings panel, right side). About 1.3 GB download.
- Same pattern for every model: **chop the clip into pieces → build → fit → score every test clip → evaluate.**

## 0. Setup and download (start this first, it takes a few minutes)

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import keras
from keras import layers
from IPython.display import Audio, display

keras.utils.set_random_seed(42)
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

Keras 3.15.1 | backend: tensorflow


In [ ]:
import urllib.request, zipfile

MACHINE = "valveEmu"     # the machine we study
OTHER = "bearingEmu"     # a different machine, only used by the CNN in Section 8

DATA_DIR = os.getcwd()   # data lands next to this notebook
URL = "https://zenodo.org/records/19336329/files/dev_{}.zip"

for m in [MACHINE, OTHER]:
    if not os.path.isdir(os.path.join(DATA_DIR, m, "train")):
        zpath = os.path.join(DATA_DIR, f"dev_{m}.zip")
        print(f"Downloading {m} (~600 MB) to {DATA_DIR} ...")
        urllib.request.urlretrieve(URL.format(m), zpath)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(DATA_DIR)   # creates DATA_DIR/<machine>/train and /test
        os.remove(zpath)

train_files = sorted(glob.glob(os.path.join(DATA_DIR, MACHINE, "train", "*.wav")))
test_files  = sorted(glob.glob(os.path.join(DATA_DIR, MACHINE, "test", "*.wav")))
other_files = sorted(glob.glob(os.path.join(DATA_DIR, OTHER, "train", "*.wav")))
print(len(train_files), "train |", len(test_files), "test |", len(other_files), "other-machine train")

## 1. The hook: can YOU hear the broken valve?

🤔 **Vote before you listen:** which clip is the broken valve? A or B? How sure are you?

In [ ]:
clip_a = [f for f in test_files if "anomaly" in f][0]
clip_b = [f for f in test_files if "normal" in f][0]
print("Clip A"); display(Audio(clip_a))
print("Clip B"); display(Audio(clip_b))

💡 Hard for humans. Now imagine 10,000 valves in a factory, 24 hours a day, and nobody has ever recorded a broken one. That's the job.

## 2. The data: the file name IS the label

In [ ]:
for f in [train_files[0], train_files[-1], test_files[0], test_files[-1]]:
    print(os.path.basename(f))

In [ ]:
names = [os.path.basename(f) for f in test_files]
y_test = np.array([1 if "anomaly" in n else 0 for n in names])          # 1 = broken
domain_test = np.array(["target" if "target" in n else "source" for n in names])

print("train: normal only ->", sum("anomaly" in f for f in train_files), "anomalies")
pd.crosstab(domain_test, y_test, rownames=["domain"], colnames=["anomaly?"])

🔍 Training has **zero** broken clips. Test has 100 normal + 100 broken. `source` = usual conditions (990 train clips), `target` = new conditions (only 10 train clips).

## 3. Turning sound into a picture: the log-mel spectrogram

In [ ]:
y, sr = librosa.load(train_files[0], sr=None, mono=False)
print("shape:", y.shape, "| sample rate:", sr, "| seconds:", y.shape[1] / sr)

🔍 2 rows = 2 microphones (0 = near the machine, 1 = far away). 16,000 numbers per second. Too many and too wiggly for a model. So we make a picture.

In [ ]:
def load_logmel(path, channel=0):
    y, sr = librosa.load(path, sr=None, mono=False)   # keep both mics, native 16 kHz
    y = y[channel, : sr * 10]                          # pick one mic, first 10 seconds
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=1024, hop_length=512, n_mels=128)
    return librosa.power_to_db(mel).T                  # (313 time steps, 128 mel bands), in dB

x = load_logmel(train_files[0])
print(x.shape)

In [ ]:
plt.figure(figsize=(12, 3))
for i, (title, f) in enumerate([("normal", clip_b), ("broken", clip_a)]):
    plt.subplot(1, 2, i + 1)
    plt.imshow(load_logmel(f).T, origin="lower", aspect="auto", cmap="magma")
    plt.title(title); plt.xlabel("time step"); plt.ylabel("mel band (low to high pitch)")
plt.show()

Now load every clip. ⏱️ About 1-2 minutes on Kaggle.

In [ ]:
%%time
X_train = np.array([load_logmel(f) for f in train_files])
X_test  = np.array([load_logmel(f) for f in test_files])
X_other = np.array([load_logmel(f) for f in other_files])

mu, sd = X_train.mean(), X_train.std()          # learned from TRAIN only
X_train, X_test, X_other = (X_train - mu) / sd, (X_test - mu) / sd, (X_other - mu) / sd
print(X_train.shape, X_test.shape, X_other.shape)

## 4. How do we score a model? Anomaly score, AUC and pAUC

In [ ]:
from sklearn.metrics import roc_auc_score
from scipy.stats import hmean

def evaluate(name, scores):
    # official DCASE rules: AUC per domain (its normals + all anomalies), pAUC on everything up to 10% false alarms
    src = (domain_test == "source") | (y_test == 1)
    tgt = (domain_test == "target") | (y_test == 1)
    row = {"model": name,
           "AUC_source": 100 * roc_auc_score(y_test[src], scores[src]),
           "AUC_target": 100 * roc_auc_score(y_test[tgt], scores[tgt]),
           "pAUC": 100 * roc_auc_score(y_test, scores, max_fpr=0.1)}
    print({k: round(v, 1) if k != "model" else v for k, v in row.items()})
    return row

🤔 **Before running:** a model that gives every clip a random score. What AUC? A) 0  B) 50  C) 100

In [ ]:
results, scores = [], {}
rng = np.random.default_rng(0)
row = evaluate("Random guess", rng.random(len(y_test)))

🤔 **No-model baseline:** score = how loud the clip is. Will louder mean broken?

In [ ]:
s = X_test.mean(axis=(1, 2))                 # average loudness of each test clip
results.append(evaluate("Loudness (no model)", s)); scores["Loudness (no model)"] = s

In [ ]:
# The number to beat: official DCASE 2026 baseline, valveEmu, development set (mean of 5 runs)
official = [{"model": "OFFICIAL baseline AE (MSE)",         "AUC_source": 67.74, "AUC_target": 68.78, "pAUC": 55.08},
            {"model": "OFFICIAL baseline AE (Mahalanobis)", "AUC_source": 56.60, "AUC_target": 56.50, "pAUC": 50.20}]
pd.DataFrame(official)

## 5. Model 1: Dense autoencoder (ANN) on 5-frame slices

Same recipe as the official baseline: chop each clip into slices of 5 time steps, flatten (5 × 128 = 640 numbers), squeeze through 8 numbers, rebuild.

In [ ]:
def pieces(X, frames):
    # chop every clip into non-overlapping pieces of `frames` time steps
    n = X.shape[1] // frames * frames
    return X[:, :n].reshape(-1, frames, 128)

print("one clip :", X_train[0].shape)
print("5-frame  :", pieces(X_train, 5).shape)
print("32-frame :", pieces(X_train, 32).shape)

In [ ]:
dense_ae = keras.Sequential([
    keras.Input(shape=(5, 128)),
    layers.Flatten(),                                  # 640 numbers
    layers.Dense(128, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(8, activation="relu"),                # bottleneck
    layers.Dense(128, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(5 * 128),
    layers.Reshape((5, 128)),
])
dense_ae.summary()

In [ ]:
P = pieces(X_train, 5)
dense_ae.compile(optimizer="adam", loss="mse")
dense_ae.fit(P, P, epochs=20, batch_size=256, verbose=2)     # input = target: rebuild yourself

In [ ]:
def ae_scores(model, X, frames):
    P = pieces(X, frames)
    R = model.predict(P, batch_size=512, verbose=0)
    err = ((P - R) ** 2).mean(axis=(1, 2))          # rebuild error of each piece
    return err.reshape(len(X), -1).mean(axis=1)     # average the pieces -> one score per clip

s = ae_scores(dense_ae, X_test, 5)
results.append(evaluate("Dense AE", s)); scores["Dense AE"] = s

💡 Score = how badly the model rebuilds the clip. It only ever practised on healthy valves, so a broken one should be rebuilt badly.

## 6. Model 2: Convolutional autoencoder on 32-frame patches

Each piece is now a small **picture**: 32 time steps × 128 mel bands, about 1 second. Conv layers look for local patterns (clicks, hums) anywhere in it.

In [ ]:
conv_ae = keras.Sequential([
    keras.Input(shape=(32, 128)),
    layers.Reshape((32, 128, 1)),                                              # add channel axis
    layers.Conv2D(16, 3, strides=2, padding="same", activation="relu"),        # 16 x 64
    layers.Conv2D(32, 3, strides=2, padding="same", activation="relu"),        # 8 x 32
    layers.Conv2D(8, 3, strides=2, padding="same", activation="relu"),         # 4 x 16 x 8 = bottleneck
    layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu"),
    layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu"),
    layers.Conv2DTranspose(1, 3, strides=2, padding="same"),                   # back to 32 x 128 x 1
    layers.Reshape((32, 128)),
])
conv_ae.summary()

In [ ]:
P = pieces(X_train, 32)
conv_ae.compile(optimizer="adam", loss="mse")
conv_ae.fit(P, P, epochs=20, batch_size=64, verbose=2)

In [ ]:
s = ae_scores(conv_ae, X_test, 32)
results.append(evaluate("Conv AE", s)); scores["Conv AE"] = s

## 7. Model 3: Sequence autoencoder, LSTM vs GRU

Same 32-step pieces, read as a **sequence** of 32 frames. Encoder reads all 32 and squeezes them into 64 numbers; decoder writes the 32 frames back.

In [ ]:
lstm_ae = keras.Sequential([
    keras.Input(shape=(32, 128)),
    layers.LSTM(64),                           # read 32 frames -> 64 numbers (bottleneck)
    layers.RepeatVector(32),                   # hand those 64 numbers to each of 32 output steps
    layers.LSTM(64, return_sequences=True),    # write 32 steps
    layers.Dense(128),                         # each step -> 128 mel bands
])
lstm_ae.compile(optimizer="adam", loss="mse")
start = time.time()
lstm_ae.fit(pieces(X_train, 32), pieces(X_train, 32), epochs=10, batch_size=64, verbose=2)
print(f"LSTM: {lstm_ae.count_params():,} weights, {time.time() - start:.0f} s")

🤔 **Before running:** swap one word, LSTM → GRU. Fewer weights or more? Better or worse score?

In [ ]:
gru_ae = keras.Sequential([
    keras.Input(shape=(32, 128)),
    layers.GRU(64),
    layers.RepeatVector(32),
    layers.GRU(64, return_sequences=True),
    layers.Dense(128),
])
gru_ae.compile(optimizer="adam", loss="mse")
start = time.time()
gru_ae.fit(pieces(X_train, 32), pieces(X_train, 32), epochs=10, batch_size=64, verbose=2)
print(f"GRU: {gru_ae.count_params():,} weights, {time.time() - start:.0f} s")

In [ ]:
s = ae_scores(lstm_ae, X_test, 32)
results.append(evaluate("LSTM seq AE", s)); scores["LSTM seq AE"] = s
s = ae_scores(gru_ae, X_test, 32)
results.append(evaluate("GRU seq AE", s)); scores["GRU seq AE"] = s

## 8. Model 4: CNN classifier, "is this my valve?"

A classifier needs two classes, but we have only healthy valves. Trick: use **another machine's** healthy sounds as class 1.
The CNN learns *what makes a valve sound like a valve*. Anomaly score = how much a test clip does **not** sound like our valve.

In [ ]:
P_valve, P_other = pieces(X_train, 32), pieces(X_other, 32)
Xc = np.concatenate([P_valve, P_other])
yc = np.concatenate([np.zeros(len(P_valve)), np.ones(len(P_other))])     # 0 = our valve, 1 = other machine
print(Xc.shape, "| valve pieces:", len(P_valve), "| other pieces:", len(P_other))

In [ ]:
cnn = keras.Sequential([
    keras.Input(shape=(32, 128)),
    layers.Reshape((32, 128, 1)),
    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),     # probability "not my valve"
])
cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
cnn.fit(Xc, yc, epochs=5, batch_size=64, shuffle=True, verbose=2)

In [ ]:
p = cnn.predict(pieces(X_test, 32), verbose=0)[:, 0]     # P("not my valve") for every piece
s = p.reshape(len(X_test), -1).mean(axis=1)              # one score per clip
results.append(evaluate("CNN valve-vs-other", s)); scores["CNN valve-vs-other"] = s

## 9. The ablation table: one dataset, four architectures, one fair comparison

In [ ]:
table = pd.DataFrame(results + official).set_index("model")
table["hmean"] = hmean(table[["AUC_source", "AUC_target", "pAUC"]], axis=1)   # DCASE-style single number
table.round(1).sort_values("hmean", ascending=False)

In [ ]:
table[["AUC_source", "AUC_target", "pAUC"]].plot.barh(figsize=(8, 5))
plt.axvline(50, color="gray", linestyle="--", label="coin toss")
plt.xlabel("score (%)"); plt.legend(); plt.show()

## 10. Failure examples: listen to the worst misses

In [ ]:
ours = table.drop(index=[r["model"] for r in official] + ["Loudness (no model)"])
best = ours["hmean"].idxmax()
s = scores[best]
false_alarm = np.argmax(np.where(y_test == 0, s, -np.inf))   # healthy clip with the HIGHEST score
missed      = np.argmin(np.where(y_test == 1, s, np.inf))    # broken clip with the LOWEST score
print("best model:", best)
for title, i in [("FALSE ALARM (healthy, scored high)", false_alarm), ("MISSED (broken, scored low)", missed)]:
    print(title, "|", names[i], "| score", round(s[i], 3)); display(Audio(test_files[i]))

In [ ]:
plt.figure(figsize=(12, 3))
for k, (title, i) in enumerate([("false alarm", false_alarm), ("missed anomaly", missed)]):
    plt.subplot(1, 2, k + 1)
    plt.imshow(X_test[i].T, origin="lower", aspect="auto", cmap="magma"); plt.title(f"{title}: {domain_test[i]}")
plt.show()

In [ ]:
plt.hist(s[y_test == 0], bins=30, alpha=0.6, label="healthy")
plt.hist(s[y_test == 1], bins=30, alpha=0.6, label="broken")
plt.xlabel(f"anomaly score ({best})"); plt.legend(); plt.show()

## 11. Deliberate breaks

### Break 1: the score is upside down
🤔 We flip the sign of the Dense AE score. What happens to AUC?

In [ ]:
row = evaluate("Dense AE, sign flipped", -scores["Dense AE"])

### Break 2: feed the wrong shape
🤔 Give the Dense AE (trained on 5-frame slices) 32-frame pieces. **Read the error first.**

In [ ]:
dense_ae.predict(pieces(X_test, 32))

### Break 3: use the far microphone
🤔 Same trained Dense AE, but we feed it the far mic (more factory noise). Better, same or worse?

In [ ]:
%%time
X_test_far = (np.array([load_logmel(f, channel=1) for f in test_files]) - mu) / sd
row = evaluate("Dense AE, far mic at test", ae_scores(dense_ae, X_test_far, 5))

## 12. Gradio demo: upload a sound, get a verdict

In [ ]:
train_s = ae_scores(dense_ae, X_train, 5)
threshold = np.percentile(train_s, 90)       # same rule as the official baseline: flag the top 10% of healthy scores

def check_machine(path):
    x = (load_logmel(path) - mu) / sd
    s = ae_scores(dense_ae, x[None], 5)[0]
    verdict = "ANOMALY: send an engineer" if s > threshold else "normal"
    return f"{verdict}  (score {s:.3f}, threshold {threshold:.3f})"

print(check_machine(clip_a)); print(check_machine(clip_b))

In [ ]:
!pip install -q gradio
import gradio as gr

demo = gr.Interface(fn=check_machine, inputs=gr.Audio(type="filepath"), outputs="text",
                    title="Listen to the Machine", examples=[[clip_a], [clip_b]])
demo.launch(share=True)

## 13. Wrap-up and homework
**Remember:** train on normal only → score = "how surprised is the model" → judge by AUC/pAUC against the official baseline.

**Homework for the 2-3 week project (pick at least 3):**
1. Run the same notebook on a second machine type: change `MACHINE` (try `fan` or `gearboxEmu`) and add rows to the table.
2. Train the Dense AE for 100 epochs and with overlapping slices, like the official baseline. Does the gap close?
3. Retrain on the far mic (`channel=1`) instead of only testing on it. Then try averaging both mics.
4. Change the bottleneck (8 → 2, 32, 128) in the Dense AE. Make a small table.
5. Use more "other machines" in Section 8 (all 6 other types). Does the CNN get better?
6. Look at source vs target AUC. Why is target harder? Try giving the 10 target clips more weight.
7. Put your best model in the Gradio demo and share the link.